# Skattning av ett efterfrågesystem för hushållsenergi med Seemingly Unrelated Regression

## Sammanfattning

Ett regionalt energibolag skattar ett efterfrågesystem med två ekvationer för **hushållsenergi** — budgetandelar för el och naturgas som funktioner av eget pris, korspris och hushållsinkomst — med hjälp av **PROC SYSLIN** och metoden seemingly-unrelated-regression (SUR). De två andelsekvationerna delar en gemensam hushållsbudget, så deras fel är korskorrelerade; SUR skattar ekvationerna gemensamt för att utnyttja den korrelationen, vilket ger teckenkoherenta egen- och korspriseffekter samt levererar den korsekvationskovarians och de restriktionstest som en efterfrågeanalytiker behöver.

## Datakällor

| Datamängd | Rader | Granularitet | Nyckelvariabler | Beskrivning |
|---------|------|-------|---------------|-------------|
| `energy` | 60 | en rad per månatlig marknadsobservation | `month`, `lp_elec`, `lp_gas`, `lincome`, `w_elec`, `w_gas` | Syntetisk månatlig panel för hushållens energimarknad. `lp_elec`/`lp_gas` är log-reala priser på el och naturgas; `lincome` är log-real hushållsinkomst; `w_elec`/`w_gas` är budgetandelar för utgifter på el och naturgas, genererade från en känd efterfrågestruktur av AIDS-typ plus korrelerat korsekvationsbrus. |

# Skattning av ett efterfrågesystem för hushållsenergi med Seemingly Unrelated Regression

Ett regionalt gas- och elbolag vill förstå hur hushåll omfördelar sin energibudget mellan **el** och **naturgas** när relativpriser och inkomst förändras. Det naturliga ramverket är ett **efterfrågesystem**: en uppsättning budgetandelsekvationer som skattas gemensamt.

Två egenskaper gör gemensam skattning till rätt verktyg:

- Andelsekvationerna bygger på en gemensam hushållsbudget, så deras fel är **korskorrelerade** — när ett hushåll spenderar mer på el spenderar det mindre på gas. Att skatta ekvationerna tillsammans med **seemingly unrelated regression (SUR)** utnyttjar den korrelationen för effektivitet.
- Ekonomisk teori inför **linjära restriktioner** (adderbarhet, homogenitet, symmetri) som en systemskattare kan tillämpa direkt.

`PROC SYSLIN` med `SUR`-alternativet är byggd för precis denna situation. Den anpassar varje andelsekvation, skattar korsekvationens felkovarians och anpassar om systemet med den kovariansen för att leverera effektiva, teorikoherenta elasticiteter — tillsammans med korsmodellens kovariansmatris och gemensamma restriktionstest.

I denna notebook:

1. Genererar vi en syntetisk månatlig energimarknadspanel från en känd andelsstruktur.
2. Anpassar vi ett andelssystem med två ekvationer med SUR.
3. Jämför vi den gemensamma SUR-anpassningen med ekvation-för-ekvation OLS.
4. Inför vi en homogenitetsrestriktion och läser dess gemensamma F-test.
5. Extraherar vi koefficienttabellen för elasticitetsrapportering.

## Steg 1 — Generera en syntetisk energimarknadspanel

Vi simulerar 60 månatliga observationer av ett litet **efterfrågesystem för energi med två varor** (el och naturgas). Datagenererande processen är en lineariserad budgetandelsmodell av AIDS-typ:

```
w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1
w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2
```

Vi bygger medvetet in **korsekvationskorrelation**: när hushåll spenderar mer på el spenderar de mindre på gas, så `e1` och `e2` är negativt korrelerade. En gemensam prisfaktor på energimarknaden driver också båda log-priserna tillsammans. Det är dessa egenskaper som gör gemensam SUR-skattning fördelaktig framför att anpassa varje ekvation för sig.

In [1]:
data energy;
   CALL streaminit(70123);

   /* True structural coefficients (linearized AIDS share system) */
   a1  = 0.55;  g11 =  0.12; g12 = -0.08; b1 = -0.030;
   a2  = 0.45;  g21 = -0.08; g22 =  0.10; b2 = -0.025;

   GÖR month = 1 TILL 60;
      /* A common energy-market price factor drives BOTH prices,
         creating the collinearity that makes the problem ill-posed. */
      energy_factor = rand('NORMAL', 0, 0.15);

      lp_elec = 4.6 + energy_factor + rand('NORMAL', 0, 0.05);
      lp_gas  = 4.2 + energy_factor + rand('NORMAL', 0, 0.05);
      lincome = 10.8 + 0.004*month + rand('NORMAL', 0, 0.06);

      /* Negatively correlated cross-equation errors (budget tradeoff) */
      shock = rand('NORMAL', 0, 0.010);
      e1 =  shock + rand('NORMAL', 0, 0.006);
      e2 = -shock + rand('NORMAL', 0, 0.006);

      w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1;
      w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2;

      UTDATA;
   SLUT;

   BEHÅLL month lp_elec lp_gas lincome w_elec w_gas;
KÖR;

PROCEDUR MEDELVÄRDEN data=energy n mean std MIN MAX maxdec=3;
   VARIABEL w_elec w_gas lp_elec lp_gas lincome;
KÖR;

                                                  The MEANS Procedure

 Variable        N           Mean     Std Dev     Minimum     Maximum
 --------------------------------------------------------------------
 W_ELEC         60          0.440       0.011       0.417       0.464
 W_GAS          60          0.228       0.014       0.198       0.256
 LP_ELEC        60          4.617       0.142       4.354       4.911
 LP_GAS         60          4.211       0.162       3.818       4.566
 LINCOME        60         10.916       0.088      10.723      11.174
 --------------------------------------------------------------------



NOTE: DATA energy


NOTE: Wrote energy (60 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Steg 2 — Grundläggande SUR-skattning av efterfrågesystemet

Vi skattar båda andelsekvationerna gemensamt. `SUR`-alternativet talar om för `PROC SYSLIN` att skatta korsekvationens felkovarians och använda den för en effektiv anpassning med feasible-GLS. Två märkta `MODEL`-satser (`elec` och `gas`) definierar systemet; var och en regresserar en budgetandel på de två log-priserna och log-inkomsten.

SYSLIN rapporterar varje ekvations parameterskattningar och, i slutet, **Cross-Model Covariance Matrix** — den skattade kovariansen mellan de två ekvationernas fel som SUR utnyttjar.

In [2]:
PROCEDUR syslin data=energy ON;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;
KÖR;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Steg 3 — Jämför med ekvation-för-ekvation OLS

För att se vad SUR ger oss anpassar vi om samma två ekvationer med minstakvadratmetoden (standardmetoden, en ekvation i taget). OLS ignorerar korsekvationens felkorrelation, så den är konsistent men mindre effektiv än SUR när den korrelationen är skild från noll — vilket den är här per konstruktion.

Att jämföra de två koefficienttabellerna visar hur skattningarna och deras standardfel förskjuts när systemstrukturen tas i beaktande.

In [3]:
PROCEDUR syslin data=energy ols;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;
KÖR;


                       The SYSLIN Procedure

                  OLS Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.155544       5.15      0.0000
  LP_ELEC        1      0.112463    0.021989       5.11      0.0000
  LP_GAS         1     -0.097938    0.019300      -5.07      0.0000
  LINCOME        1     -0.042842    0.013702      -3.13      0.0028

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (OLS)
NOTE: PROC SYSLIN completed.


## Steg 4 — Inför ekonomisk teori och testa den

Efterfrågeteori säger att nominella pris-/inkomsteffekter bör följa **homogenitet av grad noll**: att skala upp alla priser och inkomst lämnar den reala efterfrågan oförändrad, så pris- och inkomstkoefficienterna inom en ekvation summerar till noll. Vi inför det på elandelen med en `RESTRICT`-sats.

SYSLIN anpassar om systemet under bivillkoret och rapporterar ett **Restriction Results** F-test av huruvida restriktionen är förenlig med data — ett direkt, gemensamt test av homogenitetshypotesen.

In [4]:
PROCEDUR syslin data=energy ON;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;

   /* Homogeneity on the electricity share equation:
      price and income coefficients sum to zero. */
   restrict lp_elec + lp_gas + lincome = 0;
KÖR;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Steg 5 — Fånga skattningar för elasticitetsrapportering

Till en slutrapport sparar vi de konvergerade koefficienterna med `OUTEST=`. Den resulterande datamängden bär en rad per ekvation med intercept- och lutningsskattningar plus anpassningsstatistik, vilka matar in i bolagets beräkningar av egen- och korspriselasticiteter vid stickprovets medelandelar. Vi skriver ut den för dokumentationen.

In [5]:
PROCEDUR syslin data=energy ON outest=demand_est;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;
KÖR;

PROCEDUR SKRIV data=demand_est noobs;
   TITEL "SUR demand-system coefficient estimates";
KÖR;
TITEL;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: Wrote OUTEST= dataset demand_est (2 rows).
NOTE: PROC SYSLIN completed.
NOTE: PROC PRINT data=demand_est

NOTE: PROC PRINT completed: 2 observations printed, 12 variables


## Tolkning av resultaten

**Teckenkoherens.** SUR-skattningarna återger den substitutionsstruktur som är inbyggd i data. I elandelsekvationen är **koefficienten för eget log-pris positiv** (`LP_ELEC` = 0.112) medan **korspriskoefficienten är negativ** (`LP_GAS` = -0.098); i gasandelsekvationen speglas mönstret (eget `LP_GAS` = 0.114, kors `LP_ELEC` = -0.075). Varje egen- och korspriseffekt är statistiskt signifikant (varje `Pr > |t|` under 0.005), så substitutionstecknen är fast identifierade snarare än artefakter av en brusig anpassning.

**SUR utnyttjar korsekvationskorrelationen.** **Cross-Model Covariance Matrix** visar ett negativt värde utanför diagonalen (-0.000068): hushåll avväger elutgifter mot gasutgifter, precis som konstruerat. Eftersom den kovariansen är skild från noll är gemensam SUR-skattning effektivare än att anpassa varje ekvation för sig — SUR-standardfelen i Steg 2 är genomgående något mindre än ekvation-för-ekvation OLS-standardfelen i Steg 3 (till exempel faller standardfelet för gas `LP_ELEC` från 0.0264 under OLS till 0.0255 under SUR), medan punktskattningarna är oförändrade. Denna snävare inferens är vinsten med att modellera systemet snarare än två separata regressioner.

**Teorin håller.** Att införa **homogenitet av grad noll** på elandelen (pris- och inkomstkoefficienterna summerar till noll) via `RESTRICT` gav ett Restriction Results F-test på 2.14 med `Pr > F` = 0.149. Restriktionen **förkastas inte**, så homogenitetsförutsägelsen från konsumentefterfrågeteorin är förenlig med detta stickprov — bolaget kan med tillförsikt ta med de begränsade, teorikoherenta skattningarna i rapporteringen.

**Operativ användning.** `OUTEST=`-tabellen sparar en rad per ekvation med intercept- och lutningsskattningarna och anpassningsstatistiken. Dessa koefficienter omvandlas direkt till egen- och korspriselasticiteter och en inkomstelasticitet (utgiftselasticitet) vid stickprovets medelandelar (`W_ELEC` ~ 0.44, `W_GAS` ~ 0.23 från Steg 1), vilket ger bolaget en försvarbar, teorikonsistent grund för efterfrågeprognoser och taxekonstruktionsscenarier.